In [13]:
# ==============================================================================
# BLOCK 1: IMPORTS AND EXPANDED GRANDMASTER CORPUS DOWNLOAD
# ==============================================================================
!pip install -q python-chess requests tqdm

import io
import time
import zipfile
import requests
import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

print(f"[INFO] Initializing BOZON v0.4 Dual-Head (Policy + Value) Network")
print(f"[INFO] Computing Device: {device}")

PGN_MENTOR_PLAYERS = [
    "Carlsen",
    "Kasparov",
    "Fischer",
    "Tal",
    "Karpov",
    "Anand",
    "Kramnik",
    "Nakamura"
]

def fetch_pgn_mentor_corpus(players: list) -> str:
    """Downloads and extracts GM archives from pgnmentor.com."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }
    combined_pgn = []

    for player in players:
        url = f"https://www.pgnmentor.com/players/{player}.zip"
        print(f"[INFO] Downloading {player}.zip...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
                for filename in zf.namelist():
                    if filename.endswith(".pgn"):
                        pgn_text = zf.read(filename).decode("utf-8", errors="ignore")
                        combined_pgn.append(pgn_text)
                        print(f"[SUCCESS] Extracted '{filename}' ({len(pgn_text) // 1024:,} KB)")
        except Exception as exc:
            print(f"[WARNING] Failed to fetch {player}: {exc}")

    full_corpus = "\n\n".join(combined_pgn)
    print(f"[INFO] Total Corpus Size: {len(full_corpus) / (1024 * 1024):.2f} MB")
    return full_corpus


pgn_data = fetch_pgn_mentor_corpus(PGN_MENTOR_PLAYERS)

[INFO] Initializing BOZON v0.4 Dual-Head (Policy + Value) Network
[INFO] Computing Device: cuda
[INFO] Downloading Carlsen.zip...
[SUCCESS] Extracted 'Carlsen.pgn' (5,598 KB)
[INFO] Downloading Kasparov.zip...
[SUCCESS] Extracted 'Kasparov.pgn' (1,318 KB)
[INFO] Downloading Fischer.zip...
[SUCCESS] Extracted 'Fischer.pgn' (537 KB)
[INFO] Downloading Tal.zip...
[SUCCESS] Extracted 'Tal.pgn' (1,421 KB)
[INFO] Downloading Karpov.zip...
[SUCCESS] Extracted 'Karpov.pgn' (2,330 KB)
[INFO] Downloading Anand.zip...
[SUCCESS] Extracted 'Anand.pgn' (2,780 KB)
[INFO] Downloading Kramnik.zip...
[SUCCESS] Extracted 'Kramnik.pgn' (2,949 KB)
[INFO] Downloading Nakamura.zip...
[SUCCESS] Extracted 'Nakamura.pgn' (7,592 KB)
[INFO] Total Corpus Size: 23.95 MB


In [14]:
# ==============================================================================
# BLOCK 2: 18-CHANNEL BOARD ENCODING & GAME-LEVEL SPLIT DATASET
# ==============================================================================
def board_to_tensor(board: chess.Board) -> np.ndarray:
    """
    Converts a chess.Board into an (18, 8, 8) float32 tensor.
    Channels 0-5   : White pieces (P, N, B, R, Q, K)
    Channels 6-11  : Black pieces (p, n, b, r, q, k)
    Channel 12     : Turn indicator (1.0 = White, 0.0 = Black)
    Channel 13     : Legal destination squares mask
    Channel 14-15  : White Kingside / Queenside Castling Rights
    Channel 16-17  : Black Kingside / Queenside Castling Rights
    """
    tensor = np.zeros((18, 8, 8), dtype=np.float32)

    for square, piece in board.piece_map().items():
        row = 7 - (square // 8)
        col = square % 8
        channel = piece.piece_type - 1 + (6 if piece.color == chess.BLACK else 0)
        tensor[channel, row, col] = 1.0

    if board.turn == chess.WHITE:
        tensor[12, :, :] = 1.0

    for move in board.legal_moves:
        to_sq = move.to_square
        tensor[13, 7 - (to_sq // 8), to_sq % 8] = 1.0

    # Castling rights channels
    if board.has_kingside_castling_rights(chess.WHITE):
        tensor[14, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE):
        tensor[15, :, :] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):
        tensor[16, :, :] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK):
        tensor[17, :, :] = 1.0

    return tensor


def encode_move(move: chess.Move) -> int:
    return move.from_square * 64 + move.to_square


def compute_position_target_value(board: chess.Board, game_result: str) -> float:
    """
    Computes target value in [-1.0, +1.0] from the ACTIVE player's perspective.
    Blends game outcome (60%) with normalized material balance (40%) so the
    Value Head learns both immediate material safety and long-term winning chances.
    """
    # 1. Game outcome from White's perspective
    if game_result == "1-0":
        outcome_white = 1.0
    elif game_result == "0-1":
        outcome_white = -1.0
    else:
        outcome_white = 0.0

    # 2. Material balance from White's perspective
    piece_values = {
        chess.PAWN: 1.0,
        chess.KNIGHT: 3.0,
        chess.BISHOP: 3.2,
        chess.ROOK: 5.0,
        chess.QUEEN: 9.0,
        chess.KING: 0.0
    }
    material_white = 0.0
    for piece in board.piece_map().values():
        val = piece_values[piece.piece_type]
        material_white += val if piece.color == chess.WHITE else -val

    # Squash material advantage into [-1, 1] using tanh (6 points advantage ~ 0.76)
    material_score = float(np.tanh(material_white / 6.0))

    # Combined evaluation from White's perspective
    combined_white = 0.60 * outcome_white + 0.40 * material_score

    # Flip sign if it is Black's turn (Active Player Perspective)
    return combined_white if board.turn == chess.WHITE else -combined_white


class BozonDualDataset(Dataset):
    def __init__(self, states: np.ndarray, moves: np.ndarray, values: np.ndarray):
        self.states = torch.from_numpy(states)
        self.moves = torch.from_numpy(moves)
        self.values = torch.from_numpy(values).unsqueeze(1)  # Shape: (N, 1)

    def __len__(self) -> int:
        return len(self.states)

    def __getitem__(self, idx: int):
        return self.states[idx], self.moves[idx], self.values[idx]


def build_game_split_datasets(pgn_text: str, max_positions: int = 120000):
    """
    Splits data by FULL GAMES (every 10th game goes to Validation)
    to eliminate position-level data leakage between Train and Val sets.
    """
    train_max = int(max_positions * 0.90)
    val_max = max_positions - train_max

    tr_states = np.zeros((train_max, 18, 8, 8), dtype=np.float32)
    tr_moves = np.zeros((train_max,), dtype=np.int64)
    tr_values = np.zeros((train_max,), dtype=np.float32)

    va_states = np.zeros((val_max, 18, 8, 8), dtype=np.float32)
    va_moves = np.zeros((val_max,), dtype=np.int64)
    va_values = np.zeros((val_max,), dtype=np.float32)

    pgn_io = io.StringIO(pgn_text)
    game_idx = 0
    tr_idx, va_idx = 0, 0

    print(f"[INFO] Extracting {max_positions:,} positions with Game-Level Split...")

    while tr_idx < train_max or va_idx < val_max:
        game = chess.pgn.read_game(pgn_io)
        if game is None:
            break

        result = game.headers.get("Result", "*")
        if result not in ("1-0", "0-1", "1/2-1/2"):
            continue

        game_idx += 1
        is_val_game = (game_idx % 10 == 0)
        board = game.board()

        for move in game.mainline_moves():
            state_t = board_to_tensor(board)
            move_idx = encode_move(move)
            val_target = compute_position_target_value(board, result)

            if is_val_game and va_idx < val_max:
                va_states[va_idx] = state_t
                va_moves[va_idx] = move_idx
                va_values[va_idx] = val_target
                va_idx += 1
            elif not is_val_game and tr_idx < train_max:
                tr_states[tr_idx] = state_t
                tr_moves[tr_idx] = move_idx
                tr_values[tr_idx] = val_target
                tr_idx += 1

            board.push(move)

            if (tr_idx + va_idx) % 30000 == 0 and (tr_idx + va_idx) > 0:
                print(f"[PROGRESS] Train: {tr_idx:,}/{train_max:,} | Val: {va_idx:,}/{val_max:,}")

            if tr_idx >= train_max and va_idx >= val_max:
                break

    print(f"[SUCCESS] Processed {game_idx:,} distinct games.")
    print(f"[INFO] Final Train Positions: {tr_idx:,} | Unseen Game Val Positions: {va_idx:,}")

    train_ds = BozonDualDataset(tr_states[:tr_idx], tr_moves[:tr_idx], tr_values[:tr_idx])
    val_ds = BozonDualDataset(va_states[:va_idx], va_moves[:va_idx], va_values[:va_idx])
    return train_ds, val_ds


train_dataset, val_dataset = build_game_split_datasets(pgn_data, max_positions=120000)
del pgn_data

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

[INFO] Extracting 120,000 positions with Game-Level Split...
[PROGRESS] Train: 26,893/108,000 | Val: 3,107/12,000
[PROGRESS] Train: 53,913/108,000 | Val: 6,087/12,000
[PROGRESS] Train: 80,656/108,000 | Val: 9,344/12,000
[PROGRESS] Train: 108,000/108,000 | Val: 12,000/12,000
[SUCCESS] Processed 1,370 distinct games.
[INFO] Final Train Positions: 108,000 | Unseen Game Val Positions: 12,000


In [15]:
# ==============================================================================
# BLOCK 3: BOZON v0.4 DUAL-HEAD ARCHITECTURE (POLICY + VALUE)
# ==============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, channels: int, dropout_p: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.drop = nn.Dropout2d(p=dropout_p)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)


class BozonDualResNet(nn.Module):
    """
    BOZON v0.4 Dual-Head Neural Network.
    Input : (Batch, 18, 8, 8)
    Output:
      - policy_logits : (Batch, 4096) move probabilities
      - value_pred    : (Batch, 1) board evaluation in [-1.0, +1.0]
    """
    def __init__(self, num_blocks: int = 8, channels: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(18, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )

        self.res_tower = nn.Sequential(
            *[ResidualBlock(channels, dropout_p=0.1) for _ in range(num_blocks)]
        )

        # Head 1: Policy Head (Move Prediction)
        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(32 * 8 * 8, 4096)
        )

        # Head 2: Value Head (Position Evaluation [-1, +1])
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 8, kernel_size=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 1),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor):
        features = self.res_tower(self.stem(x))
        policy_logits = self.policy_head(features)
        value_pred = self.value_head(features)
        return policy_logits, value_pred


def evaluate_dual_model(model: nn.Module, dataloader: DataLoader):
    """Evaluates Policy Top-1/Top-3 Accuracy and Value Mean Squared Error (MSE)."""
    model.eval()
    pol_loss_sum = 0.0
    val_mse_sum = 0.0
    top1_correct = 0
    top3_correct = 0
    total = 0

    ce_criterion = nn.CrossEntropyLoss()
    mse_criterion = nn.MSELoss()

    with torch.no_grad():
        for states, moves, values in dataloader:
            states = states.to(device)
            moves = moves.to(device)
            values = values.to(device)

            policy_logits, value_preds = model(states)
            p_loss = ce_criterion(policy_logits, moves)
            v_loss = mse_criterion(value_preds, values)

            pol_loss_sum += p_loss.item() * states.size(0)
            val_mse_sum += v_loss.item() * states.size(0)

            _, top3_preds = torch.topk(policy_logits, k=3, dim=1)
            top1_correct += (top3_preds[:, 0] == moves).sum().item()
            top3_correct += (top3_preds == moves.unsqueeze(1)).any(dim=1).sum().item()
            total += states.size(0)

    return (
        pol_loss_sum / total,
        val_mse_sum / total,
        (top1_correct / total) * 100,
        (top3_correct / total) * 100
    )


model = BozonDualResNet(num_blocks=8, channels=128).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[INFO] BozonDualResNet (8 Residual Blocks, Dual-Head) initialized.")
print(f"[DEBUG] Trainable Parameters: {total_params:,}")

[INFO] BozonDualResNet (8 Residual Blocks, Dual-Head) initialized.
[DEBUG] Trainable Parameters: 10,848,081


In [16]:
# ==============================================================================
# BLOCK 4: JOINT MULTI-TASK TRAINING LOOP (POLICY + VALUE LOSS)
# ==============================================================================
policy_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
value_criterion = nn.MSELoss()

optimizer = optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=3e-4)
epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_val_top1 = 0.0
weights_path = "bozon_v0.4_dual.pt"

print("[INFO] Starting BOZON v0.4 Multi-Task Training...")

for epoch in range(1, epochs + 1):
    start_t = time.time()
    model.train()
    tr_pol_loss = 0.0
    tr_val_mse = 0.0
    total_train = 0

    for states, moves, values in train_loader:
        states = states.to(device)
        moves = moves.to(device)
        values = values.to(device)

        optimizer.zero_grad()
        policy_logits, value_preds = model(states)

        loss_p = policy_criterion(policy_logits, moves)
        loss_v = value_criterion(value_preds, values)

        # Combined Multi-Task Loss (Policy + weighted Value MSE)
        total_loss = loss_p + (1.5 * loss_v)

        total_loss.backward()
        optimizer.step()

        tr_pol_loss += loss_p.item() * states.size(0)
        tr_val_mse += loss_v.item() * states.size(0)
        total_train += states.size(0)

    scheduler.step()
    val_p_loss, val_v_mse, val_top1, val_top3 = evaluate_dual_model(model, val_loader)
    elapsed = time.time() - start_t

    saved_marker = ""
    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        torch.save(model.state_dict(), weights_path)
        saved_marker = " [BEST SAVED]"

    print(
        f"[EPOCH {epoch:02d}/{epochs:02d}] ({elapsed:.1f}s) | "
        f"Tr Pol: {tr_pol_loss/total_train:.3f} | Val Pol: {val_p_loss:.3f} | "
        f"Val MSE: {val_v_mse:.4f} | Top-1: {val_top1:.2f}% | Top-3: {val_top3:.2f}%{saved_marker}"
    )

model.load_state_dict(torch.load(weights_path, map_location=device))
weight_checksum = float(next(model.parameters()).sum().item())

print(f"\n[SUCCESS] Best v0.4 Dual-Head weights saved to '{weights_path}'")
print(f"[METRIC] Best Unseen-Game Validation Top-1 Accuracy: {best_val_top1:.2f}%")
print(f"[VERIFICATION] First Layer Weight Sum Checksum: {weight_checksum:.6f}")

[INFO] Starting BOZON v0.4 Multi-Task Training...
[EPOCH 01/15] (23.3s) | Tr Pol: 5.410 | Val Pol: 3.789 | Val MSE: 0.2259 | Top-1: 19.79% | Top-3: 35.17% [BEST SAVED]
[EPOCH 02/15] (22.6s) | Tr Pol: 3.625 | Val Pol: 3.330 | Val MSE: 0.2222 | Top-1: 24.65% | Top-3: 42.80% [BEST SAVED]
[EPOCH 03/15] (22.3s) | Tr Pol: 2.966 | Val Pol: 3.235 | Val MSE: 0.2125 | Top-1: 25.57% | Top-3: 43.97% [BEST SAVED]
[EPOCH 04/15] (22.4s) | Tr Pol: 2.539 | Val Pol: 3.249 | Val MSE: 0.2436 | Top-1: 26.54% | Top-3: 44.94% [BEST SAVED]
[EPOCH 05/15] (22.3s) | Tr Pol: 2.231 | Val Pol: 3.331 | Val MSE: 0.2390 | Top-1: 26.75% | Top-3: 45.20% [BEST SAVED]
[EPOCH 06/15] (22.3s) | Tr Pol: 2.005 | Val Pol: 3.420 | Val MSE: 0.2431 | Top-1: 27.44% | Top-3: 45.61% [BEST SAVED]
[EPOCH 07/15] (22.3s) | Tr Pol: 1.817 | Val Pol: 3.518 | Val MSE: 0.2467 | Top-1: 27.56% | Top-3: 45.61% [BEST SAVED]
[EPOCH 08/15] (22.3s) | Tr Pol: 1.679 | Val Pol: 3.580 | Val MSE: 0.2531 | Top-1: 28.21% | Top-3: 45.72% [BEST SAVED]
[EPOCH

In [17]:
# ==============================================================================
# BLOCK 5: TEST POLICY AND VALUE HEADS ON ADVANTAGEOUS POSITIONS
# ==============================================================================
def analyze_position(board: chess.Board, model: nn.Module, top_k: int = 3):
    model.eval()
    state_tensor = torch.tensor(
        board_to_tensor(board), dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        policy_logits, value_pred = model(state_tensor)
        policy_logits = policy_logits.squeeze(0)
        active_eval = value_pred.item()  # [-1.0, +1.0] for active player

    # Convert evaluation to White's perspective and Win Probability %
    white_eval = active_eval if board.turn == chess.WHITE else -active_eval
    white_win_prob = (white_eval + 1.0) * 50.0

    legal_mask = torch.full((4096,), float("-inf"), device=device)
    legal_moves_dict = {}
    for move in board.legal_moves:
        idx = encode_move(move)
        legal_mask[idx] = 0.0
        if idx not in legal_moves_dict or move.promotion == chess.QUEEN:
            legal_moves_dict[idx] = move

    probs = torch.softmax(policy_logits + legal_mask, dim=0)
    k = min(top_k, len(legal_moves_dict))
    top_probs, top_indices = torch.topk(probs, k)

    candidates = [
        (legal_moves_dict[idx.item()].uci(), board.san(legal_moves_dict[idx.item()]), prob.item() * 100)
        for prob, idx in zip(top_probs, top_indices)
    ]
    return white_eval, white_win_prob, candidates


# Test 1: Balanced starting position
b1 = chess.Board()
eval1, win1, moves1 = analyze_position(b1, model)
print(f"[TEST 1 - Starting Board] Eval: {eval1:+.3f} | White Win Chance: {win1:.1f}%")
print(f" -> Top Move: {moves1[0][1]} ({moves1[0][2]:.1f}%)\n")

# Test 2: White has an extra Queen (Black lost Queen on d8)
b2 = chess.Board("rnb1kbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2")
eval2, win2, moves2 = analyze_position(b2, model)
print(f"[TEST 2 - White Up a Queen] Eval: {eval2:+.3f} | White Win Chance: {win2:.1f}%")
print(f" -> Top Move: {moves2[0][1]} ({moves2[0][2]:.1f}%)")

[TEST 1 - Starting Board] Eval: +0.036 | White Win Chance: 51.8%
 -> Top Move: e4 (46.5%)

[TEST 2 - White Up a Queen] Eval: +0.065 | White Win Chance: 53.2%
 -> Top Move: Nf3 (97.2%)
